# 🎙️ OmniVoice Studio — Voice_Verge Backend

| | |
|---|---|
| **Engine** | OmniVoice (`k2-fsa/OmniVoice`) |
| **Repo** | https://github.com/k2-fsa/OmniVoice |
| **HuggingFace** | https://huggingface.co/k2-fsa/OmniVoice |
| **Languages** | 600+ |
| **Backend** | FastAPI exposed via ngrok tunnel |
| **Project folder** | `Voice_Verge.zip` uploaded directly to Colab |\n

---

## ✅ Before You Start

1. Upload `Voice_Verge.zip` to Colab via the Files pane on the left.\n
2. Set runtime: **Runtime → Change runtime type → T4 GPU**
3. Run cells **top to bottom**, one at a time
4. After Cell 6 starts, copy the **ngrok URL** and set it in `frontend/.env.local`:
 ```
 VITE_API_BASE=https://xxxx.ngrok.io
 ```


## 📦 Cell 1 — Environment Setup

Mount Google Drive, verify the Voice_Verge folder, check GPU.

In [ ]:
# ── Cell 1: Environment Setup ──────────────────────────────────────────────────
import sys, os, torch, subprocess
# ── 1. Clone GitHub Repository ────────────────────────────────────────────────
if not os.path.exists('/content/TTS'):
  print("⬇️ Cloning repository...")
  subprocess.run(['git', 'clone', 'https://github.com/sunkireddy-Barath/TTS.git', '/content/TTS'])
else:
  print("✅ Repository already cloned.")

# ── 2. Locate Voice_Verge project folder ───────────────────────────────────────
DRIVE_PROJECT_PATH = '/content/TTS/Voice_Verge'

if not os.path.isdir(DRIVE_PROJECT_PATH):
  raise FileNotFoundError(
    f'\n❌  Voice_Verge not found at: {DRIVE_PROJECT_PATH}\n'
    ' Please make sure you uploaded the Voice_Verge folder directly to your Google Drive root.'
  )
print(f' Project found: {DRIVE_PROJECT_PATH}')

BACKEND_DIR  = os.path.join(DRIVE_PROJECT_PATH, 'backend')
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_PATH, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── 3. Add backend to Python path ──────────────────────────────────────────────
if BACKEND_DIR not in sys.path:
  sys.path.insert(0, BACKEND_DIR)

# ── 4. Environment variables ───────────────────────────────────────────────────
os.environ['HF_HOME']  = CHECKPOINT_DIR
os.environ['PORT']   = '8000'
os.environ['ALLOWED_ORIGINS'] = '*'

# ── 5. GPU check ───────────────────────────────────────────────────────────────
if torch.cuda.is_available():
  gpu  = torch.cuda.get_device_name(0)
  vram = torch.cuda.get_device_properties(0).total_memory / 1e9
  print(f'✅  GPU: {gpu}  ({vram:.1f} GB VRAM)')
else:
  print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

print(f'\n📁  Backend  : {BACKEND_DIR}')
print(f'📁  HF Cache : {CHECKPOINT_DIR}')
print(f'🐍  Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')
print('\n✅  Cell 1 complete.\n')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅  Google Drive mounted
✅  Project found: /content/drive/MyDrive/Voice_Verge
✅  GPU: Tesla T4  (15.6 GB VRAM)

📂  Backend  : /content/drive/MyDrive/Voice_Verge/backend
📂  HF Cache : /content/drive/MyDrive/Voice_Verge/checkpoints  (persists across sessions)
🐍  Python 3.12.13  |  PyTorch 2.11.0+cu128

✅  Cell 1 complete.


---
## 📚 Cell 2 — Install OmniVoice & Dependencies

Install the OmniVoice package from PyPI and all backend dependencies.

In [2]:
# ── Cell 2: Install OmniVoice & Dependencies ───────────────────────────────────
import subprocess, sys
# ── OmniVoice — the TTS engine ────────────────────────────────────────────────
# From PyPI (stable) — recommended
OMNIVOICE_INSTALL = 'omnivoice'
# From GitHub latest (uncomment to use):
# OMNIVOICE_INSTALL = 'git+https://github.com/k2-fsa/OmniVoice.git'
print(f'⬇️ Installing OmniVoice ({OMNIVOICE_INSTALL})…')
r = subprocess.run(
  [sys.executable, '-m', 'pip', 'install', '-q', OMNIVOICE_INSTALL],
  capture_output=True, text=True
)
print('✅  omnivoice installed.' if r.returncode == 0 else f'❌  {r.stderr[-400:]}')
# ── Rust/Cargo dependency for MossFormer2_SE_48K ────────────────────────────
print('⬇️ Installing Rust (required for MossFormer2_SE_48K)...')
subprocess.run(['apt-get', 'install', '-y', 'cargo'], capture_output=True)
# ── Backend dependencies ──────────────────────────────────────────────────────
PACKAGES = [
  'transformers>=4.40.0',
  'accelerate>=0.29.0',
  'fastapi>=0.111.0',
  'uvicorn[standard]>=0.29.0',
  'python-multipart>=0.0.9',
  'soundfile>=0.12.1',
  'pyngrok>=7.1.0',
  'pydantic>=2.7.0',
  'modelscope>=1.15.0',
  'numpy<2.0.0',
  'noisereduce',
]
print('\n📦  Installing backend dependencies…')
failed = []
for pkg in PACKAGES:
  r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', pkg],
    capture_output=True, text=True
  )
  status = '✅' if r.returncode == 0 else '❌'
  print(f' {status}  {pkg}')
  if r.returncode != 0:
    failed.append(pkg)
if failed:
  print(f'\n⚠️  {len(failed)} failed: {failed}')
else:
  print('\n✅  All dependencies installed.')


⬇️ Installing OmniVoice (omnivoice)…
✅  omnivoice installed.

📦  Installing backend dependencies…
 ✅  fastapi>=0.111.0
 ✅  uvicorn[standard]>=0.29.0
 ✅  python-multipart>=0.0.9
 ✅  soundfile>=0.12.1
 ✅  pyngrok>=7.1.0
 ✅  pydantic>=2.7.0

✅  All dependencies installed.


---
## 🧠 Cell 3 — Verify OmniVoice Import

Confirm OmniVoice is importable and backend modules are accessible.

In [3]:
# ── Cell 3: Verify OmniVoice Import ───────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)
import sys, os

BACKEND_DIR = '/content/TTS/Voice_Verge/backend'
if BACKEND_DIR not in sys.path:
  sys.path.insert(0, BACKEND_DIR)

# ── OmniVoice import test ─────────────────────────────────────────────────────
try:
  from omnivoice import OmniVoice
  print(f'✅  OmniVoice importable: {OmniVoice}')
except ValueError as ve:
  if "numpy" in str(ve).lower() or "numpy.dtype size changed" in str(ve):
    print("\n❌ NUMPY CONFLICT DETECTED!")
    print(" This happens because Colab comes with a different Numpy version.")
    print(" 👉 FIX: Go to the top menu and click: Runtime -> Restart session")
    print(" Then run THIS cell (Cell 3) again! Do not run Cell 1 & 2 again.\n")
    raise ve
except ImportError as e:
  print(f'❌  OmniVoice import failed: {e}')
  print(' Run Cell 2 first.')

# ── Backend module verification ───────────────────────────────────────────────
required = [
  'omnivoice_engine.py', 'emotion_engine.py', 'expression_engine.py',
  'tag_parser.py', 'language_router.py', 'voice_design.py',
  'voice_clone.py', 'main.py',
]
print(f'\n📄  Backend module check ({BACKEND_DIR}):')
missing = []
for f in required:
  path = os.path.join(BACKEND_DIR, f)
  if os.path.isfile(path):
    size = os.path.getsize(path)
    print(f' ✅  {f:<35} {size:>8,} bytes')
  else:
    print(f' ❌  {f}  ← MISSING')
    missing.append(f)

if missing:
  print(f'\n⚠️  WARNING: {len(missing)} backend files missing: {missing}\n')
  print('  Make sure you have uploaded these files to your Google Drive backend folder!')
  print('  If voice_design.py is missing, the server will crash in Cell 7.\n')
else:
  print('\n✅  All backend modules found.  Cell 3 complete.')

✅  OmniVoice importable: <class 'omnivoice.models.omnivoice.OmniVoice'>

📄  Backend module check (/content/drive/MyDrive/Voice_Verge/backend):
 ✅  omnivoice_engine.py        8,916 bytes
 ✅  emotion_engine.py        7,310 bytes
 ✅  expression_engine.py       3,170 bytes
 ✅  tag_parser.py          3,013 bytes
 ✅  language_router.py       9,137 bytes
 ✅  voice_design.py        6,381 bytes
 ✅  voice_clone.py         4,419 bytes
 ✅  main.py            9,285 bytes

✅  All backend modules found.  Cell 3 complete.


---
## ⬇️ Cell 4 — Download OmniVoice Model

Download `k2-fsa/OmniVoice` from HuggingFace to your Drive.
Model cache is stored in `Voice_Verge/checkpoints/` — **persists across Colab sessions**.

This cell only downloads once. Subsequent runs skip the download automatically.

In [4]:
# ── Cell 4: Download OmniVoice Model ──────────────────────────────────────────
import os
from huggingface_hub import snapshot_download

MODEL_ID   = 'k2-fsa/OmniVoice'
CHECKPOINT_DIR = '/content/TTS/Voice_Verge/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.environ['HF_HOME'] = CHECKPOINT_DIR

# Check if model already cached
model_marker = os.path.join(CHECKPOINT_DIR, 'hub', 'models--k2-fsa--OmniVoice')
if os.path.isdir(model_marker):
  print(f'✅  OmniVoice already cached at: {model_marker}')
  print(' Skipping download (delete the folder to force re-download).')
else:
  print(f'⬇️ Downloading {MODEL_ID} → {CHECKPOINT_DIR}')
  print(' This may take 5–20 minutes (model is several GB)…')

  try:
    path = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=CHECKPOINT_DIR,
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
    )
    total_gb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(path)
    for f in files
    ) / 1e9
    print(f'\n✅  Model downloaded: {path}')
    print(f' Total size: {total_gb:.2f} GB')
  except Exception as e:
    print(f'\n❌  Download failed: {e}')

print('\n✅  Cell 4 complete.')


✅  OmniVoice already cached at: /content/drive/MyDrive/Voice_Verge/checkpoints/hub/models--k2-fsa--OmniVoice
 Skipping download (delete the folder to force re-download).

✅  Cell 4 complete.


---
## 🔄 Cell 5 — Load OmniVoice Model

Load `k2-fsa/OmniVoice` into GPU memory. Takes 30–90 seconds on first run.

In [5]:
# ── Cell 5: Load OmniVoice Model ──────────────────────────────────────────────
import sys, os, time, torch

BACKEND_DIR  = '/content/TTS/Voice_Verge/backend'
CHECKPOINT_DIR = '/content/TTS/Voice_Verge/checkpoints'
for d in [BACKEND_DIR]:
  if d not in sys.path: sys.path.insert(0, d)

os.environ['HF_HOME'] = CHECKPOINT_DIR

from omnivoice_engine import OmniVoiceEngine

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'🔄  Loading OmniVoice on {device} ({dtype})…')
t0 = time.time()

engine = OmniVoiceEngine.get_instance()

try:
  engine.load(
    model_id='k2-fsa/OmniVoice',
    device_map=device,
    dtype=dtype,
  )

  elapsed = time.time() - t0
  print(f'\n✅  OmniVoice loaded in {elapsed:.1f}s')

  print(f' Device  : {engine.device}')
  print(f' Loaded  : {engine.loaded}')

  if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f' VRAM used : {used:.2f} GB / {total:.1f} GB')

  print('\n✅  Cell 5 complete.')
except KeyboardInterrupt:
  print('\n\n❌  Model loading was manually interrupted (Stop button was clicked)!')
  print('  Please run this cell again and let the download finish without stopping it.\n')
  import sys; sys.exit(1)
except Exception as e:
  print(f'\n❌  Failed to load model: {e}\n')
  import sys; sys.exit(1)

🔄  Loading OmniVoice on cuda:0 (torch.float16)…
[OmniVoice] Loading 'k2-fsa/OmniVoice' on cuda:0 (torch.float16) …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files: 0%|    | 0/13 [00:00<?, ?it/s]

Loading weights: 0%|    | 0/313 [00:00<?, ?it/s]

Loading weights: 0%|    | 0/527 [00:00<?, ?it/s]

[OmniVoice] ✅ Model loaded successfully.

✅  Model loaded in 17.9s
 Device  : cuda:0
 Loaded  : True
 Stub mode : False
 VRAM used : 2.03 GB / 15.6 GB

✅  Cell 5 complete.


# ── Cell 6: Test Audio Pipeline ───────────────────────────────────────────────
import sys, os
BACKEND_DIR = '/content/drive/MyDrive/Voice_Verge/backend'
if BACKEND_DIR not in sys.path:
  sys.path.insert(0, BACKEND_DIR)

import io, time
import soundfile as sf
from IPython.display import Audio, display
from omnivoice_engine import OmniVoiceEngine
from voice_design import VoiceDesignService
from voice_clone import VoiceCloneService

engine = OmniVoiceEngine.get_instance()
vd = VoiceDesignService(engine)
vc = VoiceCloneService(engine)

print('\n🧪 Starting Audio Tests (Voice Design)...')

# ── V1 - Emotion Only
print('\n1️⃣ V1 (Female, Happy, No Expression):')
t0 = time.time()
w1_f = vd.generate("Hello! This is a version one test with a female voice.", 'en-US', 'female', 25, 'happy', 'none', 1)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w1_f))[0], rate=24000))

# ── V2 - Emotion + Expression Dropdown
print('\n2️⃣ V2 (Female, Neutral, Giggle Expression):')
t0 = time.time()
w2_f = vd.generate("I just won the match today.", 'en-GB', 'female', 28, 'neutral', 'giggle', 2)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w2_f))[0], rate=24000))

# ── V3 - Inline Tags
print('\n3️⃣ V3 (Male, Multi-Emotion Inline Tags):')
t0 = time.time()
w3_m = vd.generate("<angry>Get out of here!</angry> <calm>Just kidding, you can stay.</calm>", 'en-US', 'male', 30, 'neutral', 'none', 3)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w3_m))[0], rate=24000))

print('\n🧪 Starting Audio Tests (Voice Cloning)...')

# We will use w2_f as the reference audio for Voice Cloning
print('\n1️⃣ V1 (Emotion Only):')
t0 = time.time()
vc1 = vc.generate("Hello, I am testing voice cloning now.", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='sad', expression='none', version=1)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc1))[0], rate=24000))

print('\n2️⃣ V2 (Emotion + Expression Dropdown):')
t0 = time.time()
vc2 = vc.generate("I can sigh like this.", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='neutral', expression='sigh', version=2)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc2))[0], rate=24000))

print('\n3️⃣ V3 (Inline Tags):')
t0 = time.time()
vc3 = vc.generate("<happy>I am so happy!</happy> <whisper>But keep it a secret.</whisper>", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='neutral', expression='none', version=3)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc3))[0], rate=24000))

print('\n✅ All tests complete!\n')

In [ ]:
# ── Cell 6: Test Audio Pipeline ───────────────────────────────────────────────
import sys, os
BACKEND_DIR = '/content/TTS/Voice_Verge/backend'
if BACKEND_DIR not in sys.path:
  sys.path.insert(0, BACKEND_DIR)

import io, time
import soundfile as sf
from IPython.display import Audio, display
from omnivoice_engine import OmniVoiceEngine
from voice_design import VoiceDesignService
from voice_clone import VoiceCloneService

engine = OmniVoiceEngine.get_instance()
vd = VoiceDesignService(engine)
vc = VoiceCloneService(engine)

print('\n🧪 Starting Audio Tests (Voice Design)...')

# ── V1 - Emotion Only
print('\n1️⃣ V1 (Female, Happy, No Expression):')
t0 = time.time()
w1_f = vd.generate("Hello! This is a version one test with a female voice.", 'en-US', 'female', 25, 'happy', 'none', 1)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w1_f))[0], rate=24000))

# ── V2 - Emotion + Expression Dropdown
print('\n2️⃣ V2 (Female, Neutral, Giggle Expression):')
t0 = time.time()
w2_f = vd.generate("I just won the match today.", 'en-GB', 'female', 28, 'neutral', 'giggle', 2)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w2_f))[0], rate=24000))

# ── V3 - Inline Tags
print('\n3️⃣ V3 (Male, Multi-Emotion Inline Tags):')
t0 = time.time()
w3_m = vd.generate("<angry>Get out of here!</angry> <calm>Just kidding, you can stay.</calm>", 'en-US', 'male', 30, 'neutral', 'none', 3)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w3_m))[0], rate=24000))

print('\n🧪 Starting Audio Tests (Voice Cloning)...')

# We will use w2_f as the reference audio for Voice Cloning
print('\n1️⃣ V1 (Emotion Only):')
t0 = time.time()
vc1 = vc.generate("Hello, I am testing voice cloning now.", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='sad', expression='none', version=1)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc1))[0], rate=24000))

print('\n2️⃣ V2 (Emotion + Expression Dropdown):')
t0 = time.time()
vc2 = vc.generate("I can sigh like this.", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='neutral', expression='sigh', version=2)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc2))[0], rate=24000))

print('\n3️⃣ V3 (Inline Tags):')
t0 = time.time()
vc3 = vc.generate("<happy>I am so happy!</happy> <whisper>But keep it a secret.</whisper>", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='neutral', expression='none', version=3)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc3))[0], rate=24000))

print('\n✅ All tests complete!\n')

---
## 🚀 Cell 7 — Launch Backend (FastAPI + ngrok)

Start the FastAPI server and create a public HTTPS tunnel.

> After the tunnel URL appears, paste it into `frontend/.env.local`:
> ```
> VITE_API_BASE=https://xxxx.ngrok.io
> ```
> Then run `npm install && npm run dev` in the `frontend/` folder.
>
> **Keep this cell running** — interrupting it stops the backend.


In [ ]:
# ── Cell 6: Test Audio Pipeline ───────────────────────────────────────────────
import sys, os
BACKEND_DIR = '/content/TTS/Voice_Verge/backend'
if BACKEND_DIR not in sys.path:
  sys.path.insert(0, BACKEND_DIR)

import io, time
import soundfile as sf
from IPython.display import Audio, display
from omnivoice_engine import OmniVoiceEngine
from voice_design import VoiceDesignService
from voice_clone import VoiceCloneService

engine = OmniVoiceEngine.get_instance()
vd = VoiceDesignService(engine)
vc = VoiceCloneService(engine)

print('\n🧪 Starting Audio Tests (Voice Design)...')

# ── V1 - Emotion Only
print('\n1️⃣ V1 (Female, Happy, No Expression):')
t0 = time.time()
w1_f = vd.generate("Hello! This is a version one test with a female voice.", 'en-US', 'female', 25, 'happy', 'none', 1)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w1_f))[0], rate=24000))

# ── V2 - Emotion + Expression Dropdown
print('\n2️⃣ V2 (Female, Neutral, Giggle Expression):')
t0 = time.time()
w2_f = vd.generate("I just won the match today.", 'en-GB', 'female', 28, 'neutral', 'giggle', 2)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w2_f))[0], rate=24000))

# ── V3 - Inline Tags
print('\n3️⃣ V3 (Male, Multi-Emotion Inline Tags):')
t0 = time.time()
w3_m = vd.generate("<angry>Get out of here!</angry> <calm>Just kidding, you can stay.</calm>", 'en-US', 'male', 30, 'neutral', 'none', 3)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(w3_m))[0], rate=24000))

print('\n🧪 Starting Audio Tests (Voice Cloning)...')

# We will use w2_f as the reference audio for Voice Cloning
print('\n1️⃣ V1 (Emotion Only):')
t0 = time.time()
vc1 = vc.generate("Hello, I am testing voice cloning now.", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='sad', expression='none', version=1)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc1))[0], rate=24000))

print('\n2️⃣ V2 (Emotion + Expression Dropdown):')
t0 = time.time()
vc2 = vc.generate("I can sigh like this.", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='neutral', expression='sigh', version=2)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc2))[0], rate=24000))

print('\n3️⃣ V3 (Inline Tags):')
t0 = time.time()
vc3 = vc.generate("<happy>I am so happy!</happy> <whisper>But keep it a secret.</whisper>", 'en-US', reference_audio_bytes=w2_f, gender='female', age=28, emotion='neutral', expression='none', version=3)
print(f' ✅ Generated in {time.time()-t0:.2f}s')
display(Audio(sf.read(io.BytesIO(vc3))[0], rate=24000))

print('\n✅ All tests complete!\n')

🚀  Starting FastAPI server on port 8000…
[OmniVoice] Loading 'k2-fsa/OmniVoice' on cuda:0 (torch.float16) …


Fetching 13 files: 0%|    | 0/13 [00:00<?, ?it/s]

Loading weights: 0%|    | 0/313 [00:00<?, ?it/s]

Loading weights: 0%|    | 0/527 [00:00<?, ?it/s]

✅  Health: {'status': 'ok', 'model_loaded': True, 'device': 'cuda:0', 'stub_mode': False}

🌐  Creating Cloudflare tunnel…


ERROR:  [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


[OmniVoice] ✅ Model loaded successfully.

═════════════════════════════════════════════════════════════════
  🎉  OMNIVOICE STUDIO BACKEND IS LIVE!
═════════════════════════════════════════════════════════════════
  PUBLIC URL  : https://demonstrate-operate-filling-modes.trycloudflare.com
═════════════════════════════════════════════════════════════════

📋  Copy this into frontend/.env.local :

  VITE_API_BASE=https://demonstrate-operate-filling-modes.trycloudflare.com


⏳  Server running. Keep this cell active.
 Runtime → Interrupt execution to stop.
